In [122]:
import os
current_dir = os.getcwd()
parent_parent_dir = os.path.dirname(os.path.dirname(current_dir))
target_folder_path = os.path.join(parent_parent_dir, "dataset_generation")

In [123]:
samples_num = 100
qubits_num = 8 # L
shots_num = 100 # K

In [124]:
dataset_path = "/heisenberg_1d/n{samples_num}|X(coupling, meas{shots})_y(energy,entropy,corrs)_q{q}.csv".format(samples_num=samples_num, shots=shots_num, q=qubits_num)

In [125]:
def read_matrix_v2(matrix):
    # '[1, 2, 3]' ----> [1, 2, 3]
    matrix = matrix.replace("[", "").replace("]", "").split(",")
    return [float(x) for x in matrix]

In [126]:
import pandas as pd 
df = pd.read_csv(target_folder_path + dataset_path)
df.head(2)

,coupling_matrix,measurement_samples,ground_state_energy,entropy,exact_correlation_matrix_xx,exact_correlation_matrix_yy,exact_correlation_matrix_zz,approx_correlation_matrix
0,"[0.0, 123.0, 39.12568176049034, 20.02056381555...","[[3, 1, 5, 1, 3, 5, 1, 0], [0, 1, 2, 3, 0, 2, ...",-388.54086,"[0.6931471798863001, 0.12601213002265294, 0.70...","[1.0000000000000002, -0.9176800360681873, 0.22...","[1.0000000000000002, -0.9176800360681873, 0.22...","[1.0000000000000002, -0.917683667436151, 0.222...","[1.0, -1.08, 0.15, -0.03, 0.18, -0.45, -0.36, ..."
1,"[0.0, 123.0, 33.65660273052492, 0.0, 0.0, 0.0,...","[[0, 1, 2, 3, 1, 3, 5, 3], [0, 4, 1, 3, 2, 3, ...",-397.10440,"[0.6931471789065607, 0.10511144722669956, 0.70...","[1.0, -0.9311131612629333, 0.2042785753591938,...","[1.0, -0.931113161262935, 0.20427857535919755,...","[1.0, -0.9311032799223612, 0.2042586834042568,...","[1.0, -0.96, -0.12, -0.18, 0.36, -0.3, -0.27, ..."


In [127]:
import numpy as np
meas_records = np.array([read_matrix_v2(x) for x in df['measurement_samples'].values])
conditions = np.array([read_matrix_v2(x) for x in df['coupling_matrix'].values])
meas_records.shape, conditions.shape

((100, 800), (100, 64))

In [128]:
meas_records = meas_records.reshape(-1, shots_num, qubits_num)
meas_records = meas_records.reshape(-1, qubits_num)
meas_records.shape

(10000, 8)

In [129]:
# copy every condition (100, 64) for 100 times, s.t. the shape is (100, 100, 64)
new_conditions = []
for i in range(samples_num):
    for j in range(shots_num):
        new_conditions.append(conditions[i])
new_conditions = np.array(new_conditions)
new_conditions.shape

(10000, 64)

#### Embedding

In [130]:
# For each training iteration, we randomly sample B rows of new_conditions and meas_records
B = 64
batch_conditions = []
batch_measures = []
sample_idx = np.random.choice(range(samples_num*shots_num), B, replace=False)
batch_conditions = new_conditions[sample_idx]
batch_measures = meas_records[sample_idx]
batch_conditions.shape, batch_measures.shape

((64, 64), (64, 8))

##### Token encoding

In [131]:
import torch
import torch.nn as nn

B, qubits_num, d = 64, 8, 128
num_embeddings = 6
embedding_dim = d
embedding_layer = nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim)

cls_token = torch.zeros(B, 1, dtype=torch.long)
token_embedding = embedding_layer(torch.cat((cls_token, torch.tensor(batch_measures).long()), dim=1))

In [132]:
batch_measures.shape, cls_token.shape

((64, 8), torch.Size([64, 1]))

In [133]:
token_embedding.shape

torch.Size([64, 9, 128])

In [134]:
from embedding import get_positional_embedding, get_token_embedding
get_positional_embedding(64, 9, 32).shape

torch.Size([9, 32])

##### Positional encoding

In [136]:
import torch
import math 

def positional_encoding(max_len, d_model):
    # max_len: the maximum length of the input sequence
    # d_model: the dimension of the model
    # return: a tensor of shape (max_len, d_model)
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


max_len = qubits_num + 1
d_model = 128
positional_embedding = positional_encoding(max_len, d_model)
positional_embedding.shape

torch.Size([9, 128])

##### Condition encoding

In [137]:
batch_conditions.shape

(64, 64)

In [144]:
## conditional encoding
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(2024)
torch.cuda.manual_seed(2024)
torch.cuda.manual_seed_all(2024)
np.random.seed(2024)

class SimpleAutoencoder(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=48, embed_dim=128):
        super(SimpleAutoencoder, self).__init__()
        
        # Define layers: single hidden layer is the embedding layer
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )
        
        # Activation function
        self.activation = nn.ReLU()

    def forward(self, x):
        # Encoding step
        x = self.activation(self.encoder(x))
        
        # Decoding step (reconstruction)
        reconstructed = self.decoder(x)
        return x, reconstructed  # Return both embedding and reconstruction

# Instantiate the model
ae = SimpleAutoencoder()
optimizer = optim.Adam(ae.parameters(), lr=0.001)
loss_function = nn.MSELoss()  # Reconstruction loss

epochs = 10000
data_loader = torch.utils.data.DataLoader(torch.tensor(batch_conditions).float(), batch_size=64, shuffle=True)

# Example training loop
for epoch in range(epochs):
    for inputs in data_loader:  # Unsupervised learning, so no labels
        optimizer.zero_grad()
        embedding, reconstructed = ae(inputs)  # Forward pass
        loss = loss_function(reconstructed, inputs)  # Calculate reconstruction error
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights
        print('Epoch %d, Loss: %.4f' % (epoch, loss.item()))


Epoch 0, Loss: 3558.4329
Epoch 1, Loss: 3531.8960
Epoch 2, Loss: 3507.7134
Epoch 3, Loss: 3482.6599
Epoch 4, Loss: 3453.6943
Epoch 5, Loss: 3419.3931
Epoch 6, Loss: 3379.4673
Epoch 7, Loss: 3333.2070
Epoch 8, Loss: 3279.1345
Epoch 9, Loss: 3215.9773
Epoch 10, Loss: 3142.7551
Epoch 11, Loss: 3058.4468
Epoch 12, Loss: 2962.0615
Epoch 13, Loss: 2852.6646
Epoch 14, Loss: 2729.4875
Epoch 15, Loss: 2591.9756
Epoch 16, Loss: 2440.1797
Epoch 17, Loss: 2274.6758
Epoch 18, Loss: 2096.6458
Epoch 19, Loss: 1908.1078
Epoch 20, Loss: 1712.3723
Epoch 21, Loss: 1514.2438
Epoch 22, Loss: 1320.1632
Epoch 23, Loss: 1137.8682
Epoch 24, Loss: 975.8547
Epoch 25, Loss: 842.2112
Epoch 26, Loss: 741.9752
Epoch 27, Loss: 673.5889
Epoch 28, Loss: 627.3491
Epoch 29, Loss: 588.2841
Epoch 30, Loss: 543.5455
Epoch 31, Loss: 488.1072
Epoch 32, Loss: 424.9891
Epoch 33, Loss: 361.4537
Epoch 34, Loss: 305.0965
Epoch 35, Loss: 261.1139
Epoch 36, Loss: 231.3838
Epoch 37, Loss: 214.6869
Epoch 38, Loss: 207.7396
Epoch 39, L

In [145]:
embedding

tensor([[101.0379,   0.0000,  70.4949,  ...,   0.0000,  64.1358,  13.5892],
        [116.4493,   0.0000,  99.4185,  ...,   0.0000,  84.4101,  56.5272],
        [ 78.5178,   0.0000,  46.9876,  ...,   0.0000,  56.0201,  32.8134],
        ...,
        [ 94.7278,   0.0000,  62.3359,  ...,   0.0000,  74.6835,  81.3940],
        [ 85.8347,   0.0000,  95.0469,  ...,   0.0000,  60.4565,  37.2636],
        [ 89.4049,   0.0000, 107.8902,  ...,   0.0000,  69.5915,  19.9521]],
       grad_fn=<ReluBackward0>)

In [146]:
# change the data type of the input data
class SimpleFFN(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=48, embed_dim=128):
        super(SimpleFFN, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),
        )
    
    def forward(self, x):
        x = self.encoder(x)
        return x

In [147]:
# load the encoder part of the trained model to new model 
encoder = SimpleFFN()

model_dict = ae.state_dict()
encoder_dict = encoder.state_dict()
state_dict = {k:v for k, v in model_dict.items() if k in encoder_dict}
encoder.load_state_dict(state_dict)

<All keys matched successfully>

In [148]:
encoder.eval()
condition_embedding = encoder(torch.tensor(batch_conditions).float())
condition_embedding.shape

torch.Size([64, 128])

In [149]:
batch_conditions.shape

(64, 64)

In [150]:
# normalize the condition_embedding to (0, 1)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
condition_embedding_norm = scaler.fit_transform(embedding.detach().numpy())
condition_embedding_norm = torch.tensor(condition_embedding_norm)

In [151]:
condition_embedding_norm.shape

torch.Size([64, 128])

##### Addition all embeddings

In [152]:
token_embedding.shape, positional_embedding.shape, condition_embedding_norm.shape

(torch.Size([64, 9, 128]), torch.Size([9, 128]), torch.Size([64, 128]))

In [153]:
condition_embedding_expanded = condition_embedding_norm.unsqueeze(1).expand(-1, qubits_num+1, -1)
positional_embedding_expanded = positional_embedding.unsqueeze(0).expand(B, -1, -1)

In [154]:
token_embedding.shape, positional_embedding_expanded.shape, condition_embedding_expanded.shape

(torch.Size([64, 9, 128]), torch.Size([64, 9, 128]), torch.Size([64, 9, 128]))

In [155]:
all_embeddings = token_embedding + positional_embedding_expanded + condition_embedding_expanded

In [156]:
all_embeddings.shape # (batch_size, seq_len, d_model)

torch.Size([64, 9, 128])

#### Unsupervised Pretrain

In [157]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [158]:
# multi-layer transformer decoder
# structure: Masked multi-head attention ---> Feed forward network ---> Layer Norm 
# Input: all_embeddings shape (B, qubits_num+1, d_model), short connect with layer norm result
# Output: shape (B, qubits_num+1, d_model)

# Implementation of Masked Multi-Head Attention
class ScaleDotProductAttention(nn.Module):
    """
    compute scale dot product attention

    Query : given sentence that we focused on (decoder)
    Key : every sentence to check relationship with Qeury(encoder)
    Value : every sentence same with Key (encoder)
    """

    def __init__(self):
        super(ScaleDotProductAttention, self).__init__()
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, q, k, v, mask=None, e=1e-12):
        # input is 4 dimension tensor
        # [batch_size, head, length, d_tensor]
        batch_size, head, length, d_tensor = k.size()

        # 1. dot product Query with Key^T to compute similarity
        k_t = k.transpose(2, 3)  # transpose
        score = (q @ k_t) / math.sqrt(d_tensor)  # scaled dot product

        # 2. apply masking (opt)
        if mask is not None:
            score = score.masked_fill(mask == 0, -10000)

        # 3. pass them softmax to make [0, 1] range
        score = self.softmax(score)

        # 4. multiply with Value
        v = score @ v

        return v, score
    
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_head):
        super(MultiHeadAttention, self).__init__()
        self.n_head = n_head
        self.attention = ScaleDotProductAttention()
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_concat = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        # 1. dot product with weight matrices
        q, k, v = self.w_q(q), self.w_k(k), self.w_v(v)

        # 2. split tensor by number of heads
        q, k, v = self.split(q), self.split(k), self.split(v)

        # 3. do scale dot product to compute similarity
        out, attention = self.attention(q, k, v, mask=mask)

        # 4. concat and pass to linear layer
        out = self.concat(out)
        out = self.w_concat(out)

        # 5. visualize attention map
        # TODO : we should implement visualization

        return out

    def split(self, tensor):
        """
        split tensor by number of head

        :param tensor: [batch_size, length, d_model]
        :return: [batch_size, head, length, d_tensor]
        """
        batch_size, length, d_model = tensor.size()

        d_tensor = d_model // self.n_head
        tensor = tensor.view(batch_size, length, self.n_head, d_tensor).transpose(1, 2)
        # it is similar with group convolution (split by number of heads)

        return tensor

    def concat(self, tensor):
        """
        inverse function of self.split(tensor : torch.Tensor)

        :param tensor: [batch_size, head, length, d_tensor]
        :return: [batch_size, length, d_model]
        """
        batch_size, head, length, d_tensor = tensor.size()
        d_model = head * d_tensor

        tensor = tensor.transpose(1, 2).contiguous().view(batch_size, length, d_model)
        return tensor
            

In [159]:
model_test = MultiHeadAttention(all_embeddings.shape[2], 8)

In [160]:
ma = MultiHeadAttention.forward(model_test, all_embeddings, all_embeddings, all_embeddings)

In [161]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-12):
        super(LayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, unbiased=False, keepdim=True)
        # '-1' means last dimension. 

        out = (x - mean) / torch.sqrt(var + self.eps)
        out = self.gamma * out + self.beta
        return out

In [162]:
class PositionwiseFeedForward(nn.Module):

    def __init__(self, d_model, hidden, drop_prob=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, hidden)
        self.linear2 = nn.Linear(hidden, d_model)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=drop_prob)

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

In [163]:
class DecoderLayer(nn.Module):

    def __init__(self, d_model, ffn_hidden, n_head, drop_prob):
        super(DecoderLayer, self).__init__()
        self.self_attention = MultiHeadAttention(d_model=d_model, n_head=n_head)
        self.norm = LayerNorm(d_model=d_model)
        self.dropout1 = nn.Dropout(p=drop_prob)


        self.ffn = PositionwiseFeedForward(d_model=d_model, hidden=ffn_hidden, drop_prob=drop_prob)
        self.dropout3 = nn.Dropout(p=drop_prob)

    def forward(self, dec, trg_mask):
        _x = dec
        x = self.self_attention(q=dec, k=dec, v=dec, mask=trg_mask)
        x = self.dropout1(x)
        x = self.ffn(x)
        x = self.dropout3(x)
        x = self.norm(x)
        return x

In [164]:
model = DecoderLayer(all_embeddings.shape[2], 128, 8, 0.1)
trg_mask = torch.ones(8, all_embeddings.shape[1], all_embeddings.shape[1])
trg_mask = torch.triu(trg_mask, diagonal=1)
trg_mask = trg_mask.masked_fill(trg_mask == 1, float('-inf'))
model.forward(all_embeddings, trg_mask).shape

torch.Size([64, 9, 128])

In [177]:
class Decoder(nn.Module):
    def __init__(self, dec_voc_size, max_len, d_model, ffn_hidden, n_head, n_layers, drop_prob, device):
        super().__init__()

        self.layers = nn.ModuleList([DecoderLayer(d_model=d_model,
                                                  ffn_hidden=ffn_hidden,
                                                  n_head=n_head,
                                                  drop_prob=drop_prob)
                                     for _ in range(n_layers)])

        self.linear = nn.Linear(d_model, dec_voc_size)

    def forward(self, trg, trg_mask):
        _trg = trg
        for layer in self.layers:
            trg = layer(trg, trg_mask)
        output = self.linear(trg + _trg)
        
        # softmax
        output = F.log_softmax(output, dim=-1)
        
        return output

In [178]:
all_embeddings.shape

torch.Size([64, 9, 128])

In [179]:
batch_size, seq_len, d_model = all_embeddings.shape
hidden_dim = 128
n_head = 8
n_layers = 6
drop_prob = 0.1

In [180]:
import math
import time

from torch import nn, optim
from torch.optim import Adam

In [169]:
trg_mask = torch.ones(8, all_embeddings.shape[1], all_embeddings.shape[1])
trg_mask = torch.triu(trg_mask, diagonal=1)
trg_mask = trg_mask.masked_fill(trg_mask == 1, float('-inf'))

In [182]:
model = Decoder(128, seq_len, d_model, hidden_dim, n_head, n_layers, drop_prob, 'cuda')
testo = model.forward(all_embeddings, trg_mask)

In [183]:
measurement_records = torch.cat((cls_token, torch.tensor(batch_measures).long()), dim=1)
measurement_records.shape, testo.shape, token_embedding.shape

(torch.Size([64, 9]), torch.Size([64, 9, 128]), torch.Size([64, 9, 128]))

In [172]:
class DecoderOnly(nn.Module):
    # all_embeddings shape (B, qubits_num+1=max_seq, d_model)
    def __init__(self, d_model, nhead, num_layers, vocab_size, max_seq_len):
        super(DecoderOnly, self).__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        output = self.transformer_decoder(tgt=tgt, memory=src, tgt_mask=tgt_mask, memory_mask=src_mask)
        output = self.fc_out(output)
        output = F.softmax(output, dim=-1)
        return output
    
    @staticmethod
    def generate_square_subsequent_mask(sz: int) -> torch.Tensor:
        """
        生成一个方形的后续掩码，用于防止模型看到未来的单词。
        
        :param sz: 掩码的大小（序列长度）
        :return: 形状为 (sz, sz) 的掩码张量
        """
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask
    

In [184]:
from torch.utils.data import DataLoader, TensorDataset
all_embeddings.view(-1, all_embeddings.size(-1)).shape

torch.Size([576, 128])

In [185]:
from torch.utils.data import DataLoader, TensorDataset
embeddings = all_embeddings
labels = F.softmax(torch.tensor(token_embedding), dim=-1)

dataset = TensorDataset(embeddings, labels)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

/tmp/ipykernel_1777228/2766571200.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = F.softmax(torch.tensor(token_embedding), dim=-1)


In [192]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#model = DecoderOnly(d_model=32, nhead=8, num_layers=6, vocab_size=512, max_seq_len=9).to(device)
model = Decoder(128, seq_len, d_model, hidden_dim, n_head, n_layers, drop_prob, 'cuda')
criterion = nn.KLDivLoss(reduction='batchmean')
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [193]:
num_epochs = 1000

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (inputs, targets) in enumerate(dataloader):
        optimizer.zero_grad()
        trg_mask = torch.ones(8, all_embeddings.shape[1], all_embeddings.shape[1])
        trg_mask = torch.triu(trg_mask, diagonal=1)
        trg_mask = trg_mask.masked_fill(trg_mask == 1, float(0))

        outputs = model(inputs.to('cpu'), trg_mask.to('cpu'))

        loss = criterion(outputs.contiguous().view(-1, all_embeddings.size(-1)), targets.contiguous().view(-1, all_embeddings.size(-1)))
 
        gradients = torch.autograd.grad(loss, model.parameters(), retain_graph=True)
        for param, grad in zip(model.parameters(), gradients):
            param.grad = grad
        # loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(dataloader):.4f}')

Epoch [1/1000], Loss: 1.1042
Epoch [2/1000], Loss: 0.8696
Epoch [3/1000], Loss: 0.7284
Epoch [4/1000], Loss: 0.6450
Epoch [5/1000], Loss: 0.5784
Epoch [6/1000], Loss: 0.5259
Epoch [7/1000], Loss: 0.4838
Epoch [8/1000], Loss: 0.4463
Epoch [9/1000], Loss: 0.4143
Epoch [10/1000], Loss: 0.3840
Epoch [11/1000], Loss: 0.3552
Epoch [12/1000], Loss: 0.3335
Epoch [13/1000], Loss: 0.3095
Epoch [14/1000], Loss: 0.2870
Epoch [15/1000], Loss: 0.2666
Epoch [16/1000], Loss: 0.2460
Epoch [17/1000], Loss: 0.2257
Epoch [18/1000], Loss: 0.2100
Epoch [19/1000], Loss: 0.1941
Epoch [20/1000], Loss: 0.1773
Epoch [21/1000], Loss: 0.1639
Epoch [22/1000], Loss: 0.1520
Epoch [23/1000], Loss: 0.1397
Epoch [24/1000], Loss: 0.1288
Epoch [25/1000], Loss: 0.1194
Epoch [26/1000], Loss: 0.1111
Epoch [27/1000], Loss: 0.1026
Epoch [28/1000], Loss: 0.0963
Epoch [29/1000], Loss: 0.0907
Epoch [30/1000], Loss: 0.0854
Epoch [31/1000], Loss: 0.0797
Epoch [32/1000], Loss: 0.0752
Epoch [33/1000], Loss: 0.0710
Epoch [34/1000], Lo

In [191]:
model.eval()
outputs = model(all_embeddings, trg_mask)
outputs.shape

torch.Size([64, 9, 32])

In [ ]:
# inputs [94, 9, 32], construct a feature aggregation layer to aggregate the features along the 2ed axis and output the matrix is [64, 32]
class FeatureAggregationLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, projection_dim):
        super(FeatureAggregationLayer, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.projection = nn.Linear(output_dim, projection_dim)
        self.tanh = nn.Tanh()
    def forward(self, x):

        x = self.fc1(x)
        x = torch.mean(x, dim=1)
        x = self.fc2(x)
        
        # Projection linear layer
        x = self.tanh(x)
        x = self.projection(x)
        return x

In [ ]:
fa = FeatureAggregationLayer(32, 16, 32, 64)
fa_outputs = fa(outputs)
fa_outputs.shape

torch.Size([64, 64])

In [9]:
# input: samples, shots, nq
# algorithm: for each sample generate ns x nq random numbers from {0,1,2,3,4,5} uniformly 
# output: samples x ns x nq matrix
import torch
import numpy as np

def generate_random_measurement_outcomes_matrix(samples, nq, shots):
    return torch.randint(0, 6, (samples, shots, nq))

def generate_random_measurement_outcomes_vector(samples, nq, shots):
    return torch.randint(0, 6, (samples, shots*nq))

In [11]:
generate_random_measurement_outcomes_vector(300, 8, 64).shape

torch.Size([300, 512])

In [16]:
ms = True 
print("sds{}".format(ms))


sdsTrue
